# Laboratorio · Tema 3 — Autoencoder denoising sobre MNIST

**Aprendizaje Profundo · CUNEF Universidad**

En este laboratorio construyes un **autoencoder** con Keras/TensorFlow y lo
usas para dos cosas:

1. **Denoising**: le damos dígitos MNIST *con ruido* y aprende a reconstruir
   la versión *limpia*, pasando la información por un **cuello de botella**
   (un código de pocas dimensiones). Sin etiquetas: el objetivo es la propia
   imagen limpia (auto-supervisión).
2. **Interpolación en el espacio latente**: codificamos dos dígitos, tomamos
   puntos intermedios entre sus códigos y los decodificamos para ver la
   transición suave de uno a otro.

> Recomendado en **Google Colab**. Activa la GPU en *Entorno de ejecución →
> Cambiar tipo de entorno de ejecución* para ir más rápido (también funciona
> en CPU con pocas épocas).

## 1. Imports

Cargamos TensorFlow/Keras, NumPy y Matplotlib. Fijamos una semilla para que
los resultados sean reproducibles.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow:', tf.__version__)

## 2. Cargar MNIST y normalizar

MNIST son 70.000 imágenes de dígitos manuscritos de 28x28 en escala de
grises. No usamos las etiquetas `y` para entrenar el autoencoder: es
aprendizaje **no supervisado**. Normalizamos los píxeles al rango `[0, 1]` y
aplanamos cada imagen a un vector de 784 valores.

In [ ]:
(x_train, _), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalizar a [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Aplanar 28x28 -> 784 (usaremos un autoencoder denso)
x_train_flat = x_train.reshape((len(x_train), 784))
x_test_flat = x_test.reshape((len(x_test), 784))

print('train:', x_train_flat.shape, ' test:', x_test_flat.shape)

## 3. Añadir ruido gaussiano

Para el objetivo *denoising* creamos una versión **ruidosa** de las imágenes:
sumamos ruido gaussiano y recortamos de nuevo a `[0, 1]`. La entrada del
autoencoder será la imagen ruidosa; el objetivo, la imagen limpia.

In [ ]:
noise_factor = 0.4

x_train_noisy = x_train_flat + noise_factor * np.random.normal(size=x_train_flat.shape)
x_test_noisy = x_test_flat + noise_factor * np.random.normal(size=x_test_flat.shape)

# Mantener los valores en [0, 1]
x_train_noisy = np.clip(x_train_noisy, 0.0, 1.0).astype('float32')
x_test_noisy = np.clip(x_test_noisy, 0.0, 1.0).astype('float32')

# Echar un vistazo: fila de arriba limpia, abajo con ruido
n = 8
plt.figure(figsize=(12, 3))
for i in range(n):
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_test_flat[i].reshape(28, 28), cmap='gray')
    plt.axis('off')
    if i == 0: ax.set_title('limpio', loc='left')
    ax = plt.subplot(2, n, n + i + 1)
    plt.imshow(x_test_noisy[i].reshape(28, 28), cmap='gray')
    plt.axis('off')
    if i == 0: ax.set_title('con ruido', loc='left')
plt.tight_layout()
plt.show()

## 4. Definir el autoencoder

Usamos la API funcional de Keras para poder reutilizar por separado el
**encoder** (imagen → código) y el **decoder** (código → imagen). El cuello
de botella es la capa `latent_dim = 32`: mucho más pequeña que los 784
valores de entrada. El decoder termina en **sigmoide** porque los píxeles
están en `[0, 1]`.

- **Encoder**: `784 → 128 → 64 → 32` (código)
- **Decoder**: `32 → 64 → 128 → 784` (sigmoide)

In [ ]:
latent_dim = 32  # tamaño del codigo (cuello de botella)

# --- Encoder ---
inputs = keras.Input(shape=(784,), name='imagen')
x = layers.Dense(128, activation='relu')(inputs)
x = layers.Dense(64, activation='relu')(x)
code = layers.Dense(latent_dim, activation='relu', name='codigo')(x)
encoder = keras.Model(inputs, code, name='encoder')

# --- Decoder (modelo independiente que parte del codigo) ---
latent_inputs = keras.Input(shape=(latent_dim,), name='codigo_in')
x = layers.Dense(64, activation='relu')(latent_inputs)
x = layers.Dense(128, activation='relu')(x)
outputs = layers.Dense(784, activation='sigmoid', name='reconstruccion')(x)
decoder = keras.Model(latent_inputs, outputs, name='decoder')

# --- Autoencoder = decoder(encoder(x)) ---
autoencoder = keras.Model(inputs, decoder(encoder(inputs)), name='autoencoder')
autoencoder.summary()

## 5. Compilar y entrenar

Optimizador **Adam** y pérdida **binary_crossentropy** (habitual cuando la
salida es sigmoide en `[0, 1]`; también valdría `mse`). Entrenamos con
`X_ruido → X_limpio`: la clave del denoising está en que la entrada lleva
ruido y el objetivo no.

> Con 20 épocas basta para ver el efecto. Súbelas si quieres más calidad.

In [ ]:
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

history = autoencoder.fit(
    x_train_noisy, x_train_flat,   # entrada ruidosa -> objetivo limpio
    epochs=20,
    batch_size=256,
    shuffle=True,
    validation_data=(x_test_noisy, x_test_flat),
)

In [ ]:
# Curva de perdida (train vs validacion)
plt.figure(figsize=(6, 4))
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.xlabel('epoca'); plt.ylabel('perdida (BCE)')
plt.title('Entrenamiento del autoencoder')
plt.legend(); plt.tight_layout(); plt.show()

## 6. Visualizar reconstrucciones

Pasamos las imágenes ruidosas de test por el autoencoder y comparamos tres
filas: **ruidoso** (entrada), **limpio** (objetivo real) y **reconstruido**
(salida del modelo). Si el denoising funciona, la tercera fila se parece a la
segunda aunque la red solo vio la primera.

In [ ]:
# Reconstruir a partir de las imagenes ruidosas de test
decoded = autoencoder.predict(x_test_noisy[:10])

n = 10
plt.figure(figsize=(15, 5))
for i in range(n):
    # fila 1: entrada con ruido
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(x_test_noisy[i].reshape(28, 28), cmap='gray'); plt.axis('off')
    if i == 0: ax.set_ylabel('ruidoso', rotation=0, labelpad=40); ax.axis('on'); ax.set_xticks([]); ax.set_yticks([])
    # fila 2: objetivo limpio
    ax = plt.subplot(3, n, n + i + 1)
    plt.imshow(x_test_flat[i].reshape(28, 28), cmap='gray'); plt.axis('off')
    if i == 0: ax.set_ylabel('limpio', rotation=0, labelpad=40); ax.axis('on'); ax.set_xticks([]); ax.set_yticks([])
    # fila 3: reconstruccion
    ax = plt.subplot(3, n, 2 * n + i + 1)
    plt.imshow(decoded[i].reshape(28, 28), cmap='gray'); plt.axis('off')
    if i == 0: ax.set_ylabel('reconstruido', rotation=0, labelpad=40); ax.axis('on'); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

## 7. Interpolación en el espacio latente

Elegimos dos dígitos distintos, los **codificamos** con el encoder para
obtener sus vectores latentes `z1` y `z2`, y generamos puntos intermedios
`z = (1 - a)·z1 + a·z2` con `a` variando de 0 a 1. Al **decodificar** cada
punto vemos cómo el modelo transita de un dígito a otro: prueba de que el
espacio latente es continuo y con sentido.

In [ ]:
# Buscar dos indices de test con dos digitos distintos (p. ej. un 2 y un 7)
idx_a = int(np.where(y_test == 2)[0][0])
idx_b = int(np.where(y_test == 7)[0][0])

# Codificar ambos digitos (usamos las versiones limpias)
z1 = encoder.predict(x_test_flat[idx_a:idx_a + 1])
z2 = encoder.predict(x_test_flat[idx_b:idx_b + 1])

# Puntos intermedios en el espacio latente
n_steps = 10
alphas = np.linspace(0.0, 1.0, n_steps)
z_interp = np.vstack([(1 - a) * z1 + a * z2 for a in alphas])

# Decodificar los codigos intermedios
decoded_interp = decoder.predict(z_interp)

plt.figure(figsize=(15, 2.2))
for i in range(n_steps):
    ax = plt.subplot(1, n_steps, i + 1)
    plt.imshow(decoded_interp[i].reshape(28, 28), cmap='gray')
    plt.axis('off')
    ax.set_title(f'a={alphas[i]:.1f}', fontsize=9)
plt.suptitle('Interpolacion en el espacio latente: de un digito a otro')
plt.tight_layout()
plt.show()

## Para llevarte

- El autoencoder aprende **sin etiquetas**: su objetivo es la propia imagen
  (limpia). Es auto-supervisión.
- El **cuello de botella** (`latent_dim = 32`) obliga a la red a quedarse con
  lo esencial; por eso puede reconstruir la señal y descartar el ruido.
- El **espacio latente** es continuo: interpolar entre dos códigos y
  decodificar produce dígitos intermedios verosímiles.

**Para experimentar:** cambia `latent_dim` (prueba 2, 8, 64) y observa cómo
afecta a la reconstrucción; sube o baja `noise_factor`; cambia los dos
dígitos de la interpolación.